### CRS Data Integration and Structure Analysis

In [1]:
# Import libraries
import pandas as pd
import os

In [2]:
path = "../data/raw/"

files = sorted([f for f in os.listdir(path) if f.endswith(".txt")])

print(files)

['CRS 2015 data.txt', 'CRS 2016 data.txt', 'CRS 2017 data.txt', 'CRS 2018 data.txt', 'CRS 2019 data.txt', 'CRS 2020 data.txt', 'CRS 2021 data.txt', 'CRS 2022 data.txt', 'CRS 2023 data.txt', 'CRS_2024_data.txt']


In [3]:
dfs = []

for file in files:
    print(f"Loading: {file}")
    
    df_temp = pd.read_csv(
        os.path.join(path, file),
        sep="|",
        encoding="utf-8",
        low_memory=False
    )
    
    # clean column names
    df_temp.columns = df_temp.columns.str.strip().str.replace('"', '')
    
    # add safety: ensure Year column exists
    if "Year" not in df_temp.columns:
        print(f"⚠ Warning: Year missing in {file}")
        
    dfs.append(df_temp)

Loading: CRS 2015 data.txt
Loading: CRS 2016 data.txt
Loading: CRS 2017 data.txt
Loading: CRS 2018 data.txt
Loading: CRS 2019 data.txt


KeyboardInterrupt: 

In [ ]:
df = pd.concat(dfs, ignore_index=True, sort=False)

print("Final Shape: ", df.shape)

In [ ]:
print(df["Year"].describe())
print(df["Year"].unique()) # check year coverage

In [ ]:
# check column consistency
print(len(df.columns))

In [ ]:
# Sanity Check
df.groupby("Year")["USD_Commitment"].count()

In [ ]:
df.columns

In [ ]:
columns_needed = [
    "Year",
    "DonorName",
    "RecipientName",
    "SectorName",
    "USD_Commitment",
    "ClimateMitigation",
    "ClimateAdaptation"
]

df = df[columns_needed]

#### Process one file at a time 

Instead of merging everything first, we:
- clean each year
- reduce columns
- save it
- then combine later

#### Clean + Reduce per FIle

In [ ]:
input_path = "../data/raw/"
output_path = "../data/processed/"

os.makedirs(output_path, exist_ok=True)

files = sorted([f for f in os.listdir(input_path) if f.endswith(".txt")])

columns_needed = [
    "Year",
    "DonorName",
    "RecipientName",
    "SectorName",
    "USD_Commitment",
    "ClimateMitigation",
    "ClimateAdaptation"
]
    
for file in files:
    print(f"Processing {file}")
    
    df = pd.read_csv(
        os.path.join(input_path, file),
        sep="|",
        encoding="utf-8",
        low_memory=False
        )
    
    # Clean columns
    df.columns = df.columns.str.strip().str.replace('"', "")
    
    # Keep only needed columns
    df = df[columns_needed]
    
    # Clean types
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df["USD_Commitment"] = pd.to_numeric(df["USD_Commitment"], errors="coerce")
    
    # Filter invalid
    df = df[df["USD_Commitment"] > 0]
    
    # Save cleaned file
    year = df["Year"].iloc[0]
    df.to_parquet(f"{output_path}/clean_{year}.parquet", index=False)

In [ ]:
# Compine cleaned files
path = "../data/processed/"

files = [f for f in os.listdir(path) if f.endswith(".parquet")]

dfs = [pd.read_parquet(os.path.join(path, f)) for f in files]

df = pd.concat(dfs, ignore_index=True)

print(df.shape)

In [ ]:
# Save Final Dataset
df.to_parquet("../data/processed/oecd_final.parquet", index=False)

Due to te size of OECD CRS dataset, I implemented a chancked ingestion and transformation pieline, processing each data year independently and consolidating optimized parquet outputs to ensure memory efficiency and scaling.